In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv
import torch_geometric.transforms as T
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
# Define the GAT model
# this implementation is credit to pytorch_geometric examples
class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads):
        super(GAT, self).__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads, dropout=0.6)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.6)

    def forward(self, data):
        h, edge_index = data.x, data.edge_index

        h = F.dropout(h, p=0.6, training=self.training)
        h = F.elu(self.conv1(h, edge_index))
        h = F.dropout(h, p=0.6, training=self.training)
        h = self.conv2(h, edge_index)

        return h

# Load the datasets
cora_dataset = Planetoid(root='/tmp/Cora', name='Cora', transform=T.NormalizeFeatures())
data = cora_dataset[0]
data = data.to(device)

In [3]:
in_feats = data.x.shape[1]
h_channels = 64
heads = 8
model = GAT(cora_dataset.num_features, h_channels, cora_dataset.num_classes, heads)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
def train(model, data, train_mask, labels):
    model.train()

    optimizer.zero_grad()
    logits = model(data.cuda())
    loss = F.cross_entropy(logits[train_mask], labels[train_mask])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

In [4]:
train(model, data, data.train_mask, data.y)

1.944279670715332

In [7]:
@torch.no_grad()
def test():
    model.eval()
    out = model(data)
    pred = out.argmax(dim=1)

    acc = (pred[data.test_mask] == data.y[data.test_mask]).sum().item() / data.test_mask.sum().item()
    return acc

for epoch in range(0, 200):
    loss = train(model, data, data.train_mask, data.y)
    acc = test()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Accuracy: {acc:.4f}')

Epoch: 000, Loss: 0.4646, Accuracy: 0.8040
Epoch: 001, Loss: 0.5283, Accuracy: 0.8090
Epoch: 002, Loss: 0.6075, Accuracy: 0.8080
Epoch: 003, Loss: 0.5924, Accuracy: 0.8100
Epoch: 004, Loss: 0.5317, Accuracy: 0.8160
Epoch: 005, Loss: 0.5836, Accuracy: 0.8190
Epoch: 006, Loss: 0.5276, Accuracy: 0.8200
Epoch: 007, Loss: 0.5725, Accuracy: 0.8180
Epoch: 008, Loss: 0.5507, Accuracy: 0.8140
Epoch: 009, Loss: 0.5498, Accuracy: 0.8120
Epoch: 010, Loss: 0.5481, Accuracy: 0.8150
Epoch: 011, Loss: 0.5842, Accuracy: 0.8130
Epoch: 012, Loss: 0.5078, Accuracy: 0.8150
Epoch: 013, Loss: 0.5733, Accuracy: 0.8130
Epoch: 014, Loss: 0.6409, Accuracy: 0.8130
Epoch: 015, Loss: 0.5979, Accuracy: 0.8110
Epoch: 016, Loss: 0.5688, Accuracy: 0.8120
Epoch: 017, Loss: 0.5027, Accuracy: 0.8120
Epoch: 018, Loss: 0.5667, Accuracy: 0.8150
Epoch: 019, Loss: 0.6328, Accuracy: 0.8180
Epoch: 020, Loss: 0.5302, Accuracy: 0.8180
Epoch: 021, Loss: 0.5481, Accuracy: 0.8170
Epoch: 022, Loss: 0.5534, Accuracy: 0.8180
Epoch: 023,

In [8]:
torch.save(model.state_dict(), 'cora_gat.pt')